In [1]:
# Cell 1: Load libraries, data and trained models for Permutation
# Permutation Importance = shuffle one feature at a time
# measure how much accuracy drops
# higher drop = more important feature
# works on ANY model (RF, CNN, AE) without wrapper complexity

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import time

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"

# load test data only
# Permutation uses test data to measure accuracy drop
X_test  = np.load(save_path + "X_test.npy")
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()

# load class names and feature names
le_classes    = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

# load trained models
import joblib
rf_model = joblib.load(save_path + "rf_model.joblib")
print("RF model loaded!")

from tensorflow import keras
cnn_model = keras.models.load_model(save_path + "cnn_model.keras")
ae_model  = keras.models.load_model(save_path + "ae_model.keras")
print("CNN model loaded!")
print("AE model loaded!")

ae_threshold = np.load(save_path + "ae_threshold.npy")[0]
benign_label = le_classes.index('benign')

# use 1000 samples for permutation
# larger sample = more stable results
# smaller than full test set for speed
sample_size = 1000
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test[sample_idx]
y_sample = y_test.iloc[sample_idx]

print(f"\nX_test: {X_test.shape}")
print(f"Sample size: {sample_size}")
print(f"Classes: {le_classes}")
print(f"Features: {len(feature_names)}")
print("\nAll loaded!")

RF model loaded!


2026-05-11 14:17:14.242487: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CNN model loaded!
AE model loaded!

X_test: (9931, 37)
Sample size: 1000
Classes: ['DoS attack', 'Replay', 'benign']
Features: 37

All loaded!


In [2]:
# Cell 2: Permutation Importance on Random Forest
#
# Algorithm for each feature j:
# 1. Calculate baseline accuracy (no shuffling)
# 2. Shuffle feature j values randomly
# 3. Calculate new accuracy after shuffling
# 4. Importance = baseline accuracy - new accuracy
# 5. Repeat K=10 times and average
# 6. Higher drop = more important feature
#
# breiman2001: original paper that invented this method

print("Running Permutation Importance on Random Forest...")

# Step 1: baseline accuracy (no shuffling)
# predict labels using original unshuffled data
y_pred_baseline = rf_model.predict(X_sample)
baseline_acc = accuracy_score(y_sample, y_pred_baseline)
print(f"Baseline RF accuracy: {baseline_acc:.4f}")

# Step 2: shuffle each feature and measure accuracy drop
n_repeats = 10  # repeat 10 times for stable estimates
rf_importances = np.zeros(X_sample.shape[1])  # one score per feature

start = time.time()

for feat_idx in range(X_sample.shape[1]):
    # store accuracy drops for each repeat
    drops = []
    for _ in range(n_repeats):
        # copy sample to avoid modifying original
        X_perm = X_sample.copy()
        # shuffle only feature j values
        # np.random.shuffle shuffles in-place (modifies array directly)
        np.random.shuffle(X_perm[:, feat_idx])
        # predict with shuffled feature
        y_pred_perm = rf_model.predict(X_perm)
        perm_acc = accuracy_score(y_sample, y_pred_perm)
        # accuracy drop = how much worse with shuffled feature
        drops.append(baseline_acc - perm_acc)
    # average drop across 10 repeats
    rf_importances[feat_idx] = np.mean(drops)
    
    # print progress every 10 features
    if (feat_idx + 1) % 10 == 0:
        print(f"  Progress: {feat_idx+1}/{X_sample.shape[1]} features done...")

elapsed_rf = round(time.time() - start, 2)
print(f"\nPermutation RF complete! Time: {elapsed_rf}s")

# get top 10 features
top_idx_rf = [int(i) for i in np.argsort(rf_importances)[::-1][:10]]

print(f"\nTop 10 Features — RF Permutation:")
print(f"{'Rank':<6} {'Feature':<35} {'Importance':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_rf):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {rf_importances[idx]:.6f}")

# plot
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_rf[::-1]],
    [rf_importances[i] for i in top_idx_rf[::-1]],
    color='steelblue'
)
plt.xlabel('Mean Accuracy Drop (Permutation Importance)')
plt.title('Permutation Importance — Random Forest (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "perm_rf_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("RF Permutation plot saved!")

Running Permutation Importance on Random Forest...
Baseline RF accuracy: 0.9920
  Progress: 10/37 features done...
  Progress: 20/37 features done...
  Progress: 30/37 features done...

Permutation RF complete! Time: 21.65s

Top 10 Features — RF Permutation:
Rank   Feature                             Importance     
--------------------------------------------------------
  1    timestamp_c                         0.481500
  2    frame.number                        0.054900
  3    frame.protocols                     0.002700
  4    ip.id                               0.002000
  5    wlan.fc.type                        0.001300
  6    wlan.seq                            0.001100
  7    wlan.duration                       0.000900
  8    ip.src                              0.000000
  9    wlan.ta                             0.000000
  10   wlan.bssid                          0.000000
RF Permutation plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1889/3180284122.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Cell 3: Permutation Importance on 1D-CNN
#
# CNN needs 3D input (samples, 37, 1)
# We must reshape before each prediction
# Otherwise same algorithm as RF
# Cell 3: Permutation Importance on 1D-CNN - FIXED

print("Running Permutation Importance on 1D-CNN...")
print("Warning: CNN is slower due to 3D reshape per prediction...")

# fix: y_sample contains numbers (0,1,2) not text labels
# CNN trained with integer labels so just use numbers directly
y_sample_enc = y_sample.astype(int)

# Step 1: baseline accuracy for CNN
X_sample_3d = X_sample.reshape(X_sample.shape[0], X_sample.shape[1], 1)
y_pred_baseline_cnn = np.argmax(
    cnn_model.predict(X_sample_3d, verbose=0), axis=1
)
baseline_acc_cnn = accuracy_score(y_sample_enc, y_pred_baseline_cnn)
print(f"Baseline CNN accuracy: {baseline_acc_cnn:.4f}")

# Step 2: shuffle each feature
cnn_importances = np.zeros(X_sample.shape[1])

start = time.time()

for feat_idx in range(X_sample.shape[1]):
    drops = []
    for _ in range(n_repeats):
        X_perm = X_sample.copy()
        np.random.shuffle(X_perm[:, feat_idx])
        X_perm_3d = X_perm.reshape(X_perm.shape[0], X_perm.shape[1], 1)
        y_pred_perm = np.argmax(
            cnn_model.predict(X_perm_3d, verbose=0), axis=1
        )
        perm_acc = accuracy_score(y_sample_enc, y_pred_perm)
        drops.append(baseline_acc_cnn - perm_acc)
    cnn_importances[feat_idx] = np.mean(drops)

    if (feat_idx + 1) % 10 == 0:
        print(f"  Progress: {feat_idx+1}/{X_sample.shape[1]} features done...")

elapsed_cnn = round(time.time() - start, 2)
print(f"\nPermutation CNN complete! Time: {elapsed_cnn}s")

top_idx_cnn = [int(i) for i in np.argsort(cnn_importances)[::-1][:10]]

print(f"\nTop 10 Features — CNN Permutation:")
print(f"{'Rank':<6} {'Feature':<35} {'Importance':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_cnn):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {cnn_importances[idx]:.6f}")

plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_cnn[::-1]],
    [cnn_importances[i] for i in top_idx_cnn[::-1]],
    color='steelblue'
)
plt.xlabel('Mean Accuracy Drop (Permutation Importance)')
plt.title('Permutation Importance — 1D-CNN (UAV Dataset)')
plt.tight_layout()
plt.savefig

Running Permutation Importance on 1D-CNN...
Baseline CNN accuracy: 0.9620
  Progress: 10/37 features done...
  Progress: 20/37 features done...
  Progress: 30/37 features done...

Permutation CNN complete! Time: 87.24s

Top 10 Features — CNN Permutation:
Rank   Feature                             Importance     
--------------------------------------------------------
  1    timestamp_c                         0.445300
  2    frame.number                        0.105600
  3    wlan.duration                       0.078400
  4    frame.len                           0.053500
  5    frame.protocols                     0.052700
  6    wlan.ra                             0.036200
  7    wlan.ta                             0.033500
  8    llc.type                            0.027900
  9    wlan.da                             0.024400
  10   wlan.seq                            0.024100


<function matplotlib.pyplot.savefig(*args, **kwargs) -> 'None'>

In [5]:
# Cell 4: Permutation Importance on Autoencoder
#
# AE uses binary accuracy (attack vs benign)
# based on reconstruction error threshold
# Same algorithm but different prediction method

print("Running Permutation Importance on Autoencoder...")

# convert labels to binary for AE evaluation
# attack=1, benign=0
y_sample_binary = (y_sample != benign_label).astype(int)

# Step 1: baseline accuracy for AE
X_pred_baseline = ae_model.predict(X_sample, batch_size=256, verbose=0)
mse_baseline = np.mean(np.power(X_sample - X_pred_baseline, 2), axis=1)
y_pred_baseline_ae = (mse_baseline > ae_threshold).astype(int)
baseline_acc_ae = accuracy_score(y_sample_binary, y_pred_baseline_ae)
print(f"Baseline AE accuracy: {baseline_acc_ae:.4f}")

# Step 2: shuffle each feature
ae_importances = np.zeros(X_sample.shape[1])

start = time.time()

for feat_idx in range(X_sample.shape[1]):
    drops = []
    for _ in range(n_repeats):
        X_perm = X_sample.copy()
        np.random.shuffle(X_perm[:, feat_idx])
        # AE prediction: reconstruct → calculate MSE → threshold
        X_pred_perm = ae_model.predict(X_perm, batch_size=256, verbose=0)
        mse_perm = np.mean(np.power(X_perm - X_pred_perm, 2), axis=1)
        y_pred_perm = (mse_perm > ae_threshold).astype(int)
        perm_acc = accuracy_score(y_sample_binary, y_pred_perm)
        drops.append(baseline_acc_ae - perm_acc)
    ae_importances[feat_idx] = np.mean(drops)
    
    if (feat_idx + 1) % 10 == 0:
        print(f"  Progress: {feat_idx+1}/{X_sample.shape[1]} features done...")

elapsed_ae = round(time.time() - start, 2)
print(f"\nPermutation AE complete! Time: {elapsed_ae}s")

# get top 10 features
top_idx_ae = [int(i) for i in np.argsort(ae_importances)[::-1][:10]]

print(f"\nTop 10 Features — AE Permutation:")
print(f"{'Rank':<6} {'Feature':<35} {'Importance':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_ae):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {ae_importances[idx]:.6f}")

# plot
plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_ae[::-1]],
    [ae_importances[i] for i in top_idx_ae[::-1]],
    color='steelblue'
)
plt.xlabel('Mean Accuracy Drop (Permutation Importance)')
plt.title('Permutation Importance — Autoencoder (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "perm_ae_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("AE Permutation plot saved!")

Running Permutation Importance on Autoencoder...
Baseline AE accuracy: 0.3000
  Progress: 10/37 features done...
  Progress: 20/37 features done...
  Progress: 30/37 features done...

Permutation AE complete! Time: 60.66s

Top 10 Features — AE Permutation:
Rank   Feature                             Importance     
--------------------------------------------------------
  1    ip.id                               0.012500
  2    tcp.seq_raw                         0.006100
  3    tcp.flags                           0.004800
  4    tcp.dstport                         0.004800
  5    tcp.srcport                         0.004700
  6    ip.flags                            0.004700
  7    tcp.options                         0.004700
  8    tcp.window_size                     0.004700
  9    tcp.hdr_len                         0.003400
  10   ip.dst                              0.002300
AE Permutation plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1889/881313488.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Cell 5: Save all Permutation results to JSON

# save RF Permutation results
perm_rf_results = {
    "model": "RandomForest",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "Permutation Importance",
    "sample_size": sample_size,
    "n_repeats": n_repeats,
    "time_seconds": elapsed_rf,
    "top_10_features": {
        feature_names[int(i)]: round(float(rf_importances[int(i)]), 6)
        for i in top_idx_rf
    }
}
with open(save_path + "perm_rf_results.json", "w") as f:
    json.dump(perm_rf_results, f, indent=2)
print("RF Permutation JSON saved!")

# save CNN Permutation results
perm_cnn_results = {
    "model": "1D-CNN",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "Permutation Importance",
    "sample_size": sample_size,
    "n_repeats": n_repeats,
    "time_seconds": elapsed_cnn,
    "top_10_features": {
        feature_names[int(i)]: round(float(cnn_importances[int(i)]), 6)
        for i in top_idx_cnn
    }
}
with open(save_path + "perm_cnn_results.json", "w") as f:
    json.dump(perm_cnn_results, f, indent=2)
print("CNN Permutation JSON saved!")

# save AE Permutation results
perm_ae_results = {
    "model": "Autoencoder",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "Permutation Importance",
    "sample_size": sample_size,
    "n_repeats": n_repeats,
    "time_seconds": elapsed_ae,
    "top_10_features": {
        feature_names[int(i)]: round(float(ae_importances[int(i)]), 6)
        for i in top_idx_ae
    }
}
with open(save_path + "perm_ae_results.json", "w") as f:
    json.dump(perm_ae_results, f, indent=2)
print("AE Permutation JSON saved!")

# save summary
perm_summary = {
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "Permutation Importance",
    "rf_top5":  [feature_names[int(i)] for i in top_idx_rf[:5]],
    "cnn_top5": [feature_names[int(i)] for i in top_idx_cnn[:5]],
    "ae_top5":  [feature_names[int(i)] for i in top_idx_ae[:5]],
    "findings": {
        "rf_perm_vs_shap":  f"RF Perm #1: {feature_names[top_idx_rf[0]]} vs SHAP #1: timestamp_c",
        "cnn_perm_vs_shap": f"CNN Perm #1: {feature_names[top_idx_cnn[0]]} vs SHAP #1: frame.number",
        "ae_perm_vs_shap":  f"AE Perm #1: {feature_names[top_idx_ae[0]]} vs SHAP #1: timestamp_c"
    }
}
with open(save_path + "perm_summary.json", "w") as f:
    json.dump(perm_summary, f, indent=2)
print("Summary JSON saved!")

print("\nAll Permutation results saved!")
print(f"\nSummary:")
print(f"  RF  Perm #1: {feature_names[top_idx_rf[0]]}")
print(f"  CNN Perm #1: {feature_names[top_idx_cnn[0]]}")
print(f"  AE  Perm #1: {feature_names[top_idx_ae[0]]}")

RF Permutation JSON saved!
CNN Permutation JSON saved!
AE Permutation JSON saved!
Summary JSON saved!

All Permutation results saved!

Summary:
  RF  Perm #1: timestamp_c
  CNN Perm #1: timestamp_c
  AE  Perm #1: ip.id
